# genomic-research Tutorial

End-to-end walkthrough: generate synthetic data → prepare → train → evaluate → analyze.

## Prerequisites
```bash
pip install genomic-research
```

## 1. Generate Synthetic Data

For this tutorial, we generate random DNA sequences. Replace with your own FASTA file.

In [ ]:
import random
import os

random.seed(42)
bases = 'ATCG'

os.makedirs('/tmp/genomic-tutorial', exist_ok=True)

with open('/tmp/genomic-tutorial/sequences.fasta', 'w') as f:
    for i in range(100):
        length = random.randint(200, 500)
        seq = ''.join(random.choice(bases) for _ in range(length))
        f.write(f'>seq_{i}\n{seq}\n')

print('Generated 100 synthetic sequences')

## 2. Initialize Experiment

Use the CLI to prepare data (tokenize, chunk, split).

In [ ]:
os.chdir('/tmp/genomic-tutorial')
!genomic-research init --fasta sequences.fasta --task pretrain --tokenizer char --max-length 128

## 3. Train the Model

Run training with a short time budget for demonstration.

In [ ]:
!GENOMIC_TIME_BUDGET=30 python train.py

## 4. Examine Results

In [ ]:
import json

with open('reports/metrics.json') as f:
    metrics = json.load(f)

print(f"Task: {metrics.get('task_type')}")
print(f"Val Score: {metrics.get('val_score'):.4f}")
print(f"Perplexity: {metrics.get('val_perplexity', 'N/A')}")
print(f"Parameters: {metrics.get('num_params')}")

## 5. View Training Curves

In [ ]:
from IPython.display import Image, display

if os.path.exists('reports/training_curve.png'):
    display(Image('reports/training_curve.png'))

if os.path.exists('reports/val_score_curve.png'):
    display(Image('reports/val_score_curve.png'))

## 6. Run Inference

In [ ]:
!python inference.py --checkpoint checkpoints/best_model.pt --fasta sequences.fasta 2>&1 | head -10

## 7. Extract Embeddings

In [ ]:
!python inference.py --checkpoint checkpoints/best_model.pt --fasta sequences.fasta --embeddings embeddings.npy

import numpy as np
embeddings = np.load('embeddings.npy')
print(f'Embeddings shape: {embeddings.shape}')

## Next Steps

- Try different architectures: edit `MODEL_TYPE` in `train.py`
- Try different tokenizers: `--tokenizer kmer --kmer-size 6`
- Increase time budget: `GENOMIC_TIME_BUDGET=300`
- See `docs/hyperparameter_guide.md` for tuning tips